# 01 — Prepare the Classification-Validation Population

## Purpose

This notebook documents how the dataset used to answer the reviewer’s questions about classification quality was constructed. It combines the private daily Parquet files, removes duplicate vacancy IDs, retains the variables required for validation, and excludes records that were not resolved by the ESCO extraction stage.

## Publication status

The original daily Parquet inputs are not included in the public replication package. The computed output of this notebook is included at `data/data-pipeline-answers/interim/classification_validation_population.parquet`, so public replication of the reviewer-answer statistics begins with notebook 02.

## Execution

Run this notebook from `notebooks/data-pipeline-answers/`. Authors with access to the private daily files should place them in the documented private-input directory before rerunning this preparation step.

**Documentation:** [Notebook guide](README.md) · [Data description](../../data/data-pipeline-answers/README.md) · [Output description](../../output/data-pipeline-answers/README.md)


## Workflow

1. Locate the private daily Parquet files.
2. Combine all daily files and retain the first occurrence of each vacancy `id`.
3. Select the eight variables required for classification validation.
4. Keep records with a non-missing `extract_type`.
5. Save the computed validation population as Parquet.

In [ ]:
from pathlib import Path

import pandas as pd

# Explicit repository-relative paths are intentional for reviewer-answer analyses.
PRIVATE_INPUT_DIR = Path("../../data/data-pipeline-answers/input/private_daily")
OUTPUT_FILE = Path(
    "../../data/data-pipeline-answers/interim/"
    "classification_validation_population.parquet"
)

## 1. Locate private daily inputs

The input directory is intentionally empty in the public package. A clear error is raised when the private daily files are unavailable.

In [ ]:
if not PRIVATE_INPUT_DIR.is_dir():
    raise FileNotFoundError(
        "Private daily input directory is unavailable. See "
        "data/data-pipeline-answers/input/private_daily/README.md."
    )

input_files = sorted(PRIVATE_INPUT_DIR.glob("*.parquet"))
if not input_files:
    raise FileNotFoundError(
        f"No private daily Parquet files found in {PRIVATE_INPUT_DIR}."
    )

print(f"Input files: {len(input_files)}")

## 2. Combine and deduplicate vacancies

Daily files are concatenated in filename order. Duplicate vacancy IDs are removed because the validation exercise evaluates unique vacancies rather than repeated daily appearances.

In [ ]:
daily_frames = [pd.read_parquet(path) for path in input_files]
combined_data = pd.concat(daily_frames, ignore_index=True)

rows_before_deduplication = combined_data.shape[0]
combined_data = combined_data.drop_duplicates(subset="id").reset_index(drop=True)

print(f"Rows before deduplication: {rows_before_deduplication:,}")
print(f"Unique vacancies: {combined_data.shape[0]:,}")

## 3. Select the validation variables

The retained fields identify each vacancy, provide cleaned text and language, record the extraction method, and contain the LLM and verified ESCO classification fields used in the manual-validation analysis.

In [ ]:
validation_columns = [
    "id",
    "clean_title",
    "clean_desc",
    "desc_lang",
    "extract_type",
    "classified_title_clean",
    "esco_title",
    "esco_code",
]

missing_columns = [
    column for column in validation_columns if column not in combined_data.columns
]
if missing_columns:
    raise KeyError(f"Required validation columns are missing: {missing_columns}")

validation_population = combined_data.loc[:, validation_columns].copy()
validation_population = validation_population[
    validation_population["extract_type"].notna()
].reset_index(drop=True)

print(f"Validation population: {validation_population.shape[0]:,} rows")
print(f"Columns retained: {validation_population.shape[1]}")

## 4. Save the computed publication file

This derived Parquet file is included in the replication package. It is the input for the next reviewer-answer notebook.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
validation_population.to_parquet(OUTPUT_FILE, index=False)

print(f"Saved: {OUTPUT_FILE}")
print(f"Shape: {validation_population.shape}")